In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!ls "/content/drive/MyDrive/MONDIcare/dataset1"


dataset


In [ ]:
!ls "/content/drive/MyDrive/MONDIcare/dataset1"


dataset


In [ ]:
!ls "/content/drive/MyDrive/MONDIcare/dataset1/dataset"


Testing  Training  Validation


In [ ]:
!ls "/content/drive/MyDrive/MONDIcare/dataset1/dataset"


Testing  Training  Validation


In [ ]:
import tensorflow as tf

# Paths using the nested 'dataset' folder
train_dir = "/content/drive/MyDrive/MONDIcare/dataset1/dataset/Training"
val_dir   = "/content/drive/MyDrive/MONDIcare/dataset1/dataset/Validation"
test_dir  = "/content/drive/MyDrive/MONDIcare/dataset1/dataset/Testing"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Load datasets
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Check class names
print("Classes in Training:", train_ds.class_names)
print("Training batches:", len(train_ds))
print("Validation batches:", len(val_ds))
print("Testing batches:", len(test_ds))


Found 3251 files belonging to 3 classes.
Found 416 files belonging to 3 classes.
Found 405 files belonging to 3 classes.
Classes in Training: ['Early_Blight', 'Healthy', 'Late_Blight']
Training batches: 102
Validation batches: 13
Testing batches: 13


In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping

# Preprocess datasets for MobileNetV2
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

# Load MobileNetV2 with pretrained ImageNet weights
base_model = MobileNetV2(input_shape=(224, 224, 3),
                         include_top=False,
                         weights='imagenet')

# Freeze the base model
base_model.trainable = False

# Build the model
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(3, activation='softmax')  # 3 classes
])

# Compile
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Early stopping
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop]
)

# Evaluate on test set
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc:.2f}")


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 1228s 12s/step - accuracy: 0.5903 - loss: 0.8887 - val_accuracy: 0.7909 - val_loss: 0.5273
Epoch 2/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 15s 144ms/step - accuracy: 0.8095 - loss: 0.4879 - val_accuracy: 0.8005 - val_loss: 0.5221
Epoch 3/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 14s 135ms/step - accuracy: 0.8412 - loss: 0.4012 - val_accuracy: 0.8221 - val_loss: 0.4557
Epoch 4/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 13s 131ms/step - accuracy: 0.8787 - loss: 0.3306 - val_accuracy: 0.8149 - val_loss: 0.4700
Epoch 5/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 13s 131ms/step - accuracy: 0.8788 - loss: 0.3172 - val_accuracy: 0.7909 - val_loss: 0.5670
Epoch 6/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 13s 131ms/step - accuracy: 0.8917 - loss: 0.2763 - val_accuracy: 0.8702 - val_loss: 0.3390
Epoch 7/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 13s 129ms/step - accuracy: 0.8984 - loss: 0.2775 - val_accuracy: 0.8654 - val_loss: 0.3524
Epoch 8/10
102/102 ━━━━━━━━━━━━━━━

In [ ]:
model.save("/content/potato_model_temp.keras")
print("✅ Model saved temporarily in Keras format")


✅ Model saved temporarily in Keras format


In [ ]:
# Unfreeze the last 50 layers of MobileNetV2
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # small learning rate
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)


Epoch 1/5
102/102 ━━━━━━━━━━━━━━━━━━━━ 50s 272ms/step - accuracy: 0.6158 - loss: 1.4951 - val_accuracy: 0.8654 - val_loss: 0.3453
Epoch 2/5
102/102 ━━━━━━━━━━━━━━━━━━━━ 14s 133ms/step - accuracy: 0.7973 - loss: 0.5843 - val_accuracy: 0.8413 - val_loss: 0.3901
Epoch 3/5
102/102 ━━━━━━━━━━━━━━━━━━━━ 14s 132ms/step - accuracy: 0.8539 - loss: 0.3966 - val_accuracy: 0.8293 - val_loss: 0.4208
Epoch 4/5
102/102 ━━━━━━━━━━━━━━━━━━━━ 14s 134ms/step - accuracy: 0.9088 - loss: 0.2530 - val_accuracy: 0.8269 - val_loss: 0.4389
Epoch 5/5
102/102 ━━━━━━━━━━━━━━━━━━━━ 14s 133ms/step - accuracy: 0.9225 - loss: 0.2144 - val_accuracy: 0.8221 - val_loss: 0.5183


In [ ]:
from tensorflow.keras import layers

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# Apply augmentation during training
train_ds_aug = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))


In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

# Path to your uploaded image
img_path = "/content/sample_leaf6.jpg"

# Load the image and resize it to match MobileNetV2 input
img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img)/255.0  # normalize pixel values to 0-1
img_array = np.expand_dims(img_array, axis=0)  # add batch dimension

# Predict
prediction = model.predict(img_array)

# Map prediction to class names
classes = ['Early_Blight', 'Healthy', 'Late_Blight']
predicted_class = classes[np.argmax(prediction)]

print(f"Predicted class: {predicted_class}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Predicted class: Late_Blight
